In [1]:
import pandas as pd
import re
from collections import Counter
from nltk.util import ngrams
from nltk.corpus import stopwords
import nltk


nltk.download('stopwords')

df = pd.read_csv(
    "test.ft.txt",
    sep="\t",
    header=None,
    names=["text"],
    engine="python",
    encoding="latin-1"
)

# Remove labels
df["review"] = df["text"].str.replace("__label__1 ", "", regex=False)
df["review"] = df["review"].str.replace("__label__2 ", "", regex=False)
# Create category labels
df["category"] = df["text"].apply(
    lambda x: "Positive" if "__label__2" in str(x) else "Negative")
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\skadi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
df

,text,review,category
0,__label__2 Great CD: My lovely Pat has one of ...,Great CD: My lovely Pat has one of the GREAT v...,Positive
1,__label__2 One of the best game music soundtra...,One of the best game music soundtracks - for a...,Positive
2,__label__1 Batteries died within a year ...: I...,Batteries died within a year ...: I bought thi...,Negative
3,"__label__2 works fine, but Maha Energy is bett...","works fine, but Maha Energy is better: Check o...",Positive
4,__label__2 Great for the non-audiophile: Revie...,Great for the non-audiophile: Reviewed quite a...,Positive
...,...,...,...
399995,__label__1 Unbelievable- In a Bad Way: We boug...,Unbelievable- In a Bad Way: We bought this Tho...,Negative
399996,"__label__1 Almost Great, Until it Broke...: My...","Almost Great, Until it Broke...: My son reciev...",Negative
399997,__label__1 Disappointed !!!: I bought this toy...,Disappointed !!!: I bought this toy for my son...,Negative
399998,__label__2 Classic Jessica Mitford: This is a ...,Classic Jessica Mitford: This is a compilation...,Positive


In [3]:

def clean_text(text):


    text = text.lower()

    # Remove special characters
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)

    # Tokenize
    words = text.split()

    # Remove stopwords
    words = [word for word in words if word not in stop_words]
      
    return words
# Apply cleaning
df["tokens"] = df["review"].apply(clean_text)


In [4]:
df

,text,review,category,tokens
0,__label__2 Great CD: My lovely Pat has one of ...,Great CD: My lovely Pat has one of the GREAT v...,Positive,"[great, cd, lovely, pat, one, great, voices, g..."
1,__label__2 One of the best game music soundtra...,One of the best game music soundtracks - for a...,Positive,"[one, best, game, music, soundtracks, game, re..."
2,__label__1 Batteries died within a year ...: I...,Batteries died within a year ...: I bought thi...,Negative,"[batteries, died, within, year, bought, charge..."
3,"__label__2 works fine, but Maha Energy is bett...","works fine, but Maha Energy is better: Check o...",Positive,"[works, fine, maha, energy, better, check, mah..."
4,__label__2 Great for the non-audiophile: Revie...,Great for the non-audiophile: Reviewed quite a...,Positive,"[great, non, audiophile, reviewed, quite, bit,..."
...,...,...,...,...
399995,__label__1 Unbelievable- In a Bad Way: We boug...,Unbelievable- In a Bad Way: We bought this Tho...,Negative,"[unbelievable, bad, way, bought, thomas, son, ..."
399996,"__label__1 Almost Great, Until it Broke...: My...","Almost Great, Until it Broke...: My son reciev...",Negative,"[almost, great, broke, son, recieved, birthday..."
399997,__label__1 Disappointed !!!: I bought this toy...,Disappointed !!!: I bought this toy for my son...,Negative,"[disappointed, bought, toy, son, loves, thomas..."
399998,__label__2 Classic Jessica Mitford: This is a ...,Classic Jessica Mitford: This is a compilation...,Positive,"[classic, jessica, mitford, compilation, wide,..."


In [5]:
from nltk.tokenize import word_tokenize
nltk.download('punkt_tab')

def audit_review(text):


    # Tokenize
    tokens = word_tokenize(text)

    # Remove short tokens
    tokens = [t for t in tokens if len(t) > 1]

    total_tokens = len(tokens)

    stopword_tokens = [t for t in tokens if t in stop_words]
    informative_tokens = [t for t in tokens if t not in stop_words]

    stopword_count = len(stopword_tokens)
    informative_count = len(informative_tokens)

    # Ratios
    stopword_ratio = (
        stopword_count / total_tokens
        if total_tokens > 0 else 0
    )

    informative_ratio = (
        informative_count / total_tokens
        if total_tokens > 0 else 0
    )

    return pd.Series([
        total_tokens,
        stopword_count,
        informative_count,
        stopword_ratio,
        informative_ratio
    ])



df[
    [
        "total_tokens",
        "stopword_count",
        "informative_count",
        "stopword_ratio",
        "informative_ratio"
    ]
] = df["review"].apply(audit_review)



category_summary = df.groupby("category")[
    [
        "total_tokens",
        "stopword_count",
        "informative_count",
        "stopword_ratio",
        "informative_ratio"
    ]
].mean()


print("\nDATA QUALITY AUDIT REPORT\n")

print(category_summary)



[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\skadi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!



DATA QUALITY AUDIT REPORT

          total_tokens  stopword_count  informative_count  stopword_ratio  \
category                                                                    
Negative     79.313595        32.43194          46.881655        0.402212   
Positive     73.436395        29.42777          44.008625        0.391654   

          informative_ratio  
category                     
Negative           0.597788  
Positive           0.608346  


In [6]:
def custom_tokenizer(text):

    # Regex patterns
    patterns = [
        r'[$₹€£¥]\s?\d+(?:,\d{3})*(?:\.\d+)?',      # Currency
        r'\b\d+(?:\.\d+)?\s?(?:GB|TB|MB|GHz|MHz|W|V|mAh|inch|cm|mm|kg|g|fps|Hz|MP)\b',  # Tech specs
        r'\b\d+p\b',                               # 1080p, 720p
        r'\b[a-zA-Z]+\d+\b',                       # RTX4090, iPhone15
        r'\b\d+\.\d+\b',                           # Decimal numbers
        r'\b\w+\b'                                 # Normal words
    ]

    combined_pattern = '|'.join(patterns)

    tokens = re.findall(combined_pattern, str(text), flags=re.IGNORECASE)

    return tokens

# Apply tokenizer
df["tokens"] = df["review"].apply(custom_tokenizer)

# Show results
print(df[["review", "tokens"]].head())




                                              review  \
0  Great CD: My lovely Pat has one of the GREAT v...   
1  One of the best game music soundtracks - for a...   
2  Batteries died within a year ...: I bought thi...   
3  works fine, but Maha Energy is better: Check o...   
4  Great for the non-audiophile: Reviewed quite a...   

                                              tokens  
0  [Great, CD, My, lovely, Pat, has, one, of, the...  
1  [One, of, the, best, game, music, soundtracks,...  
2  [Batteries, died, within, a, year, I, bought, ...  
3  [works, fine, but, Maha, Energy, is, better, C...  
4  [Great, for, the, non, audiophile, Reviewed, q...  


In [7]:
bigram_counter = Counter()

for tokens in df["tokens"]:
    bigrams = ngrams(tokens, 2)
    bigram_counter.update(bigrams)

# Top 20 bigrams
top_bigrams = bigram_counter.most_common(20)

print("\nTop 20 Bigrams:\n")

for bg, count in top_bigrams:
    print(f"{' '.join(bg)} : {count}")


Top 20 Bigrams:

of the : 156403
in the : 89991
is a : 77518
this book : 72997
I have : 61089
and the : 57764
on the : 55053
to the : 53947
I was : 51835
to be : 49027
don t : 47192
for the : 46844
it s : 45938
it is : 45732
This is : 43056
and I : 42851
with the : 39266
it was : 38928
for a : 37131
the book : 36602


In [8]:
trigram_counter = Counter()

for tokens in df["tokens"]:
    trigrams = ngrams(tokens, 3)
    trigram_counter.update(trigrams)

# Top 20 trigrams
top_trigrams = trigram_counter.most_common(20)

print("\nTop 20 Trigrams:\n")

for tg, count in top_trigrams:
    print(f"{' '.join(tg)} : {count}")


Top 20 Trigrams:

I don t : 17186
one of the : 16449
This is a : 14900
a lot of : 12037
I bought this : 11946
This book is : 9931
I can t : 9720
I didn t : 9375
is a great : 8592
is one of : 8350
some of the : 7695
this is a : 7570
this book is : 7312
I had to : 6736
of the book : 6685
This is the : 6364
If you are : 6330
of the best : 6287
you want to : 6143
to be a : 6095
